# Un SGSI medido: la lectura

Cuaderno de lectura de la medición `sgsi-medido/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos): un sistema de gestión de la seguridad de la información con la estructura de ISO/IEC 27001:2022 sobre tres laboratorios publicados (accesos privilegiados, DataMart y la demo del RAG, que es el sistema de IA y se evalúa contra ISO/IEC 27090:2026). Los ficheros se leen de al lado si existen y, si no, del repositorio.

In [ ]:
import urllib.request
from pathlib import Path
import pandas as pd

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/sgsi-medido/"
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    return pd.read_csv(p if p.exists() else RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El mapa de marcos: qué cubre cada uno de las 22 categorías del CSF y cuánto se solapan

In [ ]:
cob = leer("cobertura_csf.csv").sort_values("categoria", kind="stable")
marcos = [c for c in cob.columns if c not in ("funcion", "categoria", "categoria_nombre")]
print("categorías cubiertas de 22:", {m: int((cob[m] > 0).sum()) for m in marcos})
print(cob[["categoria"] + marcos].to_string(index=False))
pares = leer("comparativa_pares.csv").sort_values(["jaccard", "marco_a", "marco_b"], kind="stable")
pares

## Los riesgos: inherente, residual y apetito

In [ ]:
print(leer("riesgos_resumen.csv").to_string(index=False))
r = leer("riesgos.csv").sort_values(["residual", "activo", "amenaza"], ascending=[False, True, True], kind="stable")
r[["activo", "laboratorio", "amenaza", "amenaza_nombre", "frecuencia", "impacto", "riesgo", "eficacia", "residual", "nivel_residual", "sobre_apetito"]].head(15)

## La declaración de aplicabilidad: 93 controles y diez amenazas de IA

In [ ]:
print(leer("soa_resumen.csv").to_string(index=False))
s90 = leer("soa_27090_detalle.csv")
print(s90[["clausula", "amenaza", "aplica", "estado", "mitigaciones_atlas"]].to_string(index=False))
leer("soa_controles.csv")[["control", "titulo", "aplica", "estado", "ens"]]

## La auditoría interna: 91 comprobaciones sobre los ficheros versionados

In [ ]:
print(leer("auditoria_resumen.csv").to_string(index=False))
print(leer("auditoria_marcos.csv").to_string(index=False))
a = leer("auditoria.csv")
a[a.resultado != "conforme"].sort_values(["resultado", "id"], kind="stable")[["id", "sistema", "resultado", "comprobacion", "iso27001", "ens", "csf"]]

## Los indicadores: objetivos y resultados clave

In [ ]:
print(leer("okr_resumen.csv").to_string(index=False))
leer("kpis.csv")[["kr", "kpi", "valor", "unidad", "meta", "estado", "fuente"]]

## El simulacro de continuidad y el plan director

In [ ]:
sim = leer("simulacro.csv")
print(sim[sim.fase.str.startswith("tiempo real") | sim.fase.str.startswith("comprobaci")][["laboratorio", "fase", "segundos_acumulados", "resultado"]].to_string(index=False))
print(leer("plan_director_resumen.csv").to_string(index=False))
leer("fuentes/plan_director.csv")[["id", "iniciativa", "prioridad", "trimestre", "horas_estimadas"]]